# Assignment - Visualization

Make sure to **read the instructions** completely. Failing to follow the instructions will have a negative impact on your grade!

In this assignment, we will explore ways to visualize the data, starting with simple plots for early exploration and build towards more interactive methods. As we want to go beyond static plots possible with for example matplotlib or seaborn, we will use the [Bokeh](https://www.bokeh.org/) library. This means the exercises **must** be implemented with Bokeh to receive points.

This assignment includes four parts (Assignmets V.1 to V.4) which provide different amounts of points. V.1 to V.3 are independent assignments, but V.4 builds on the previous assignments.
 
**Before you submit**, make sure that you are not accidentaly using any global variables. Restart the kernel (wiping all global variables) and run the code from top to bottom by clicking "Kernel" => "Restart & Run all" in the menu bar at the top.

### API documentation

note: you can find the used versions by checking the `environment.yml` file or `running conda list -n dsait4000vis` from the command line

* [Bokeh](https://docs.bokeh.org/) (also has examples)
* [Pandas](https://pandas.pydata.org/docs/)
* [NumPy](https://numpy.org/doc/stable/)
* [SciPy](https://docs.scipy.org/doc/scipy/)


### Rules
For this assignment the following rules apply:

**General**
 * The assignment should be completed in the **groups** you have already assembled for the previous assignment. Work together on all parts and do not distribute the work. Everyone is responsible for everything. That also means if there is fraud in a part it will be considered fraud by the whole group and voids the complete assignment, even if you distributed the parts of the assignment.
 * Any kind of intergroup discussion will be considered fraud and both the parties will be punished.
 * All code must be written within group. All external help, with the exception of Python/library documentation and the lecture slides, will be considered fraud.
 * Do not add any libarires to those already included notebook (i.e., `numpy`, `scipy`, `pandas`, and `bokeh`)
 * Add functionality from bokeh, like ```from bokeh.io import output_notebook```, on top of the block where you need it.
 * **Do not rename the notebook file**

**AI Use**
 * The use of AI for this assignment is permitted. If you use AI, please explain what AI was used for and how you made sure you used it responsibly in the section at the end of this file. 
 * Every group member should be able to explain all facets of the entire assignment. 

**Grading**
 * Exercise may (partially) be graded automatically, using additional hidden tests. These tests will check your solution for different (unseen) inputs.
 * If these tests pass without error (warnings are allowed) then you receive full points.
 * Do not make any assumptions on the input data (such as number or type of columns) unless specified otherwise. Doing so may result in the tests failing and thus 0 points.
 * Your grade is calculated as the sum of the achieved points and will be rounded to the closest 0.1 point.
 
**Submissions**
 * Submit your code to Brightspace
 * For this assignment you must *only* submit the notebook (`*.ipynb`) file filled in with your solution.
 * Refer to Brightspace for the exact deadline.
 * We will consider submissions that are handed in up to 3 hours after the deadline but with a penalty as follows:
 * For every completed 20 minutes past the deadline the grade will be reduced by 0.5 (e.g., with the deadline at 12:00, a submission at 12:19 will get the full grade, a submission at 12:20 will receive a 0.5 grade penalty).

In [1]:
# We do the basic setup here.
# Do not (!) change this cell! All changes will be reset during grading!

# numpy and pandas
import numpy as np
import pandas as pd

# We import a range of bokeh functionality that will likely be needed already
# if you need more, import it at the top of the corresponding cell
from bokeh.plotting import figure, gridplot, show
from bokeh.layouts import column
from bokeh.transform import factor_cmap, linear_cmap
from bokeh.models import ColumnDataSource, HoverTool, NumeralTickFormatter, BasicTicker, PrintfTickFormatter, CustomJS, Select, axes
from bokeh.models.annotations import Label
from bokeh.io import output_notebook
output_notebook() # activate Bokeh output to Jupyter notebook

# You can use this to calculate the density plot in Assignment V.3
from scipy.stats import gaussian_kde

# We load the adult data  (https://archive.ics.uci.edu/dataset/2/adult) as in previous assignments (note, we keep the headers, to use them in the pandas dataframe)
# Note: we do not change ? values to nan, but rather keep them as ? so they will be shown in the visualizations
data = pd.read_csv("datasets/Visualization/adult_all.csv")

data.head()

Loading BokehJS ...

,age,workclass,fnlwgt,education,education-num,marital-status,occupation,relationship,race,sex,capital-gain,capital-loss,hours-per-week,native-country,salary
0,39,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
1,50,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
2,38,Private,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
3,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,40,United-States,<=50K
4,28,Private,338409,Bachelors,13,Married-civ-spouse,Prof-specialty,Wife,Black,Female,0,0,40,Cuba,<=50K


### Assignment V.1 (5 points)

Getting an overview of the data distribution

In this first assignment, we want to use visualization to get a quick overview of the distribution of the different attributes/columns. This can often be helpful to make sure that we use models that match the distribution downstream or to quickly identify for example whether the data is distributed over all classes equally if the attribute is intended to be used for training a model.

A good way to quickly plot the distribution is a simple bar chart for each attribute. However, this will only allow us to look at one attribute at a time. In class we have discussed small multiples. We can use this concept to plot all attributes in a grid of plots. 

For this assignment do the following:
* Create a grid of bar charts, with one chart per column in the dataset. The attribute values shall be on the x-axis and the count of each value on the y-axis. Look at pandas `value_counts` function to get the latter.
  Hint: The `figure()` function works slightly different for numerical and categorical data.
* Make sure that the charts and axes are well labeled (for example like in the screenshot below). It will be hard to properly label the x-axis with many items (e.g., native-country). Do this as well as you can, you will likely not find a perfect solution for all attributes.
* Add a tooltip to show the values on hover (see the education chart, top-right, in the image below).

The result should look similar to the below.

[<img src="assets/barcharts.png" alt="Small Multiples showing the distributions of all columns in the dataset." width="500"/>](assets/barcharts.png)

In [2]:
from bokeh.models import NoOverlap

PLOT_SIZE = 330          # width and height of one small-multiple chart (px)
GRID_COLUMNS = 4         # charts per row
MANY_CATEGORIES = 12     # beyond this the category labels get vertical and thinned out
BAR_COLOR = "#74a9f5"
MISSING_LABEL = "(missing)"


def is_numeric(series):
    return pd.api.types.is_numeric_dtype(series) and not pd.api.types.is_bool_dtype(series)


def bin_column(series, bin_width=None):
    """Group a column into bars. Returns (bars, codes, width).

    bars: dict of equal-length arrays (x, count, plus left/right for bins);
    codes: the bar index of every row (-1 for missing or infinite numbers);
    width: bar width in x-axis units.
    Categories are sorted by count, numbers get one bar per value or, with bin_width, per occupied bin.
    """
    if not is_numeric(series):
        labels = series.astype(object).fillna(MISSING_LABEL).astype(str)
        counts = labels.value_counts()
        codes = pd.Categorical(labels, categories=counts.index).codes
        return dict(x=list(counts.index), count=counts.to_numpy()), codes.astype(np.int32), 0.8

    values = series.to_numpy(dtype=float, na_value=np.nan)
    present = np.isfinite(values)
    codes = np.full(len(values), -1, dtype=np.int32)
    if bin_width is None:
        x, codes[present], count = np.unique(values[present], return_inverse=True, return_counts=True)
        width = 0.9 * np.diff(x).min() if len(x) > 1 else 1.0
        return dict(x=x, count=count), codes, width

    start = np.floor(values[present].min() / bin_width) * bin_width
    # rounding first keeps values that sit on a bin edge (e.g. 0.3 / 0.1 = 2.9999...) in the right bin
    index = np.floor(np.round((values[present] - start) / bin_width, 9)).astype(np.int64)
    count = np.bincount(index)
    occupied = count > 0                       # empty bins cannot be drawn on a log axis
    left = start + np.flatnonzero(occupied) * bin_width
    codes[present] = (np.cumsum(occupied) - 1)[index]
    bars = dict(x=left + bin_width / 2, count=count[occupied], left=left, right=left + bin_width)
    return bars, codes, bin_width


def distribution_chart(series, bin_width=None, log_y=False):
    """Bar chart of the value counts of one column (a histogram if bin_width is given)."""
    name = str(series.name)
    numeric = is_numeric(series)
    bars, _, width = bin_column(series, bin_width)
    bottom = 0.5 if log_y else 0     # a log axis has no zero: bars start just below a count of 1
    options = {}
    if log_y:
        options.update(y_axis_type="log", y_range=(bottom, bars["count"].max() * 2))
    if not numeric:
        options["x_range"] = bars["x"]
    tooltip_value = "@left{0,0.[####]} – @right{0,0.[####]}" if bin_width else "@x"

    p = figure(title=name[:1].upper() + name[1:], width=PLOT_SIZE, height=PLOT_SIZE,
               tools="pan,wheel_zoom,box_zoom,reset,save",
               tooltips=[(name, tooltip_value), ("Count", "@count{0,0}")], **options)
    p.vbar(x="x", top="count", bottom=bottom, width=width, source=ColumnDataSource(bars),
           fill_color=BAR_COLOR, line_color="white" if bin_width else BAR_COLOR, line_width=0.5)

    p.title.text_font_size = "12pt"
    p.xaxis.axis_label = name
    p.yaxis.axis_label = "Count"
    p.yaxis.formatter = NumeralTickFormatter(format="0,0")
    if not log_y:
        p.yaxis.ticker = BasicTicker(min_interval=1)      # counts are integers: no 0.5 ticks on small data
    p.xgrid.grid_line_color = None
    p.xaxis.major_label_orientation = np.pi / 4
    if numeric:
        p.xaxis.formatter = NumeralTickFormatter(format="0,0.[####]")
    elif len(bars["x"]) > MANY_CATEGORIES:
        # too many labels to show: vertical, small, overlapping ones dropped (the tooltip names every bar)
        p.xaxis.major_label_orientation = np.pi / 2
        p.xaxis.major_label_text_font_size = "7pt"
        p.xaxis.major_label_policy = NoOverlap()
    return p


show(gridplot([distribution_chart(data[name]) for name in data.columns],
              ncols=GRID_COLUMNS, toolbar_location="above"))

#### A bar chart like this doesn't serve every attribute equally well

Look closely at the grid you just built. For some attributes, a few attribute values accounts for the overwhelming majority of the data, resulting in a few very long bars. The remaining attribute values  Such a bar chart does not allow us to see the distribution of the data in a meaningful way. 

* Identify the numerical attributes where this happens. For this task, you can select attributes by name. 
* For just those attributes, rebuild the chart as a **binned histogram**. Do not make any changes to the other bar charts. You may need to try a few different bin sizes to find a suitable one. **Justify your choice for the bin sizes in 50 words or less** in the response field below. 
* Choose an appropriate scale for the Y axis that shows the data distribution properly. **Justify your choice for the Y axis scale in 50 words or less** in the response field below.  

In [3]:
# Share of the non-missing rows taken by the most frequent value of each numerical column
dominant_share = pd.Series({name: data[name].value_counts(normalize=True).max()
                            for name in data.select_dtypes("number")}, dtype=float)
print(dominant_share.sort_values(ascending=False).round(3).to_string())

# capital-gain and capital-loss are over 90 % zeros, so one bar dwarfs all others. Only these two charts are
# rebuilt as histograms (column -> bin width); hours-per-week peaks at 47 %, its other values stay visible.
BIN_WIDTHS = {"capital-gain": 2000, "capital-loss": 100}

charts = [distribution_chart(data[name], BIN_WIDTHS.get(name), log_y=name in BIN_WIDTHS) for name in data.columns]
show(gridplot(charts, ncols=GRID_COLUMNS, toolbar_location="above"))

capital-loss      0.953
capital-gain      0.917
hours-per-week    0.467
education-num     0.323
age               0.028
fnlwgt            0.000


#### Response: Justification for binning strategy and Y axis scale 

**Bin sizes.** capital-gain: 2,000-wide bins give 16 bars instead of 123 thin ones. The 244 rows at 99,999 get their own bar. capital-loss: 100-wide bins spread 1,500–2,500, where 88% of the non-zero losses are, over ten bars.

**Y axis scale.** Logarithmic. 92% of capital-gain and 95% of capital-loss values are 0. On a linear axis the next-tallest bar is only 2% and 1% of the tallest, so you cannot see it. Log bar lengths are not proportional, so tooltips show exact counts.

### Assignment V.2 (5 points)

Visualizing possible correlations and some interactivity

In this assignment, we want to visualize the correlation between attributes. To keep this simple we only use the numerical variables here. You can use pandas `corr()` [function](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.corr.html) directly on the dataframe using the `numeric_only = True` argument. The output will contain only the results from using numerical columns. It should be noted though, that there are also ways to calculate dependencies or correlation between catagorical (e.g., [Chi-Square_test](https://en.wikipedia.org/wiki/Chi-square_test), [Cramér's V](https://en.wikipedia.org/wiki/Cramér%27s_V)) or categorical and numerical (e.g., [One-way ANOVA test](https://en.wikipedia.org/wiki/F-test#One-way_ANOVA_example)) values.

The output will be a matrix of pairwise values. It is common to visualize such data in a heatmap (images below).

* Calculate the correlation of the numerical columns using pandas `corr()`, using Pearson correlation (this is the default of the function).
* Create a heatmap showing the values. The data format output by `corr()` might not be very useful. Look for example at [pandas `stack()`](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.DataFrame.stack.html) for possible ways to transforma the data into a typical dataframe.
* A heatmap uses x and y position for the attributes and color for the value, choose a fitting colormap and **justify it below in 50 words or less**. (Note: the color chosen in the example below might or might not be good. Do **not** use the example to justify your choice.)
* Make sure the chart is legible by adding axes, legends, a color bar, or similar. It is up to you to decide what is needed.

We now want to extend our chart to be able to switch between the Pearson and Spearman correlation interactively using a dropdown menu directly in the chart.
* Add a `Select` dropdown with the two options.
* Calculate both correlation values and add a plot for each to your figure.
* Use `js_on_change` on the added Select to add code to switch between the two plots.

[<img src="assets/heatmap.png" alt="Heatmap showing correlations" width="500"/>](assets/heatmap.png)
[<img src="assets/heatmap_select.png" alt="Heatmap showing correlations with open selection widget" width="500"/>](assets/heatmap_select.png)


In [4]:
from bokeh.core.properties import field
from bokeh.models import ColorBar, FactorRange, LinearColorMapper
from bokeh.palettes import RdBu11, interp_palette

# We calculate both matrices
correlation_matrices = {
    "pearson": data.corr(numeric_only = True, method="pearson"),
    "spearman": data.corr(numeric_only = True, method="spearman")
}

# this is the default correlation method
correlation_method = "spearman"

NAN_COLOR = "#bdbdbd"        # constant columns have an undefined correlation
DARK_CELL_SHARE = 0.7        # cells beyond this share of the colour range get white text (better contrast)
MIN_FONT_PX = 7              # below this the cell labels are unreadable and left out


def to_long_format(matrix, method):
    """One row per cell (pair of attributes) of a correlation matrix."""
    names = [str(name) for name in matrix.columns]
    return pd.DataFrame({
        "x": np.tile(names, len(names)),
        "y": np.repeat(names, len(names)),
        "value": matrix.to_numpy(dtype=float).ravel(),
        "method": method.capitalize(),
    })


def color_limit(matrices):
    """Largest off-diagonal |r| of all matrices, rounded up to 0.1 (1 if there is none)."""
    off_diagonal = np.concatenate([m.to_numpy(dtype=float)[~np.eye(len(m), dtype=bool)] for m in matrices])
    strongest = np.abs(off_diagonal[np.isfinite(off_diagonal)])
    return min(np.ceil(strongest.max() * 10) / 10, 1.0) if strongest.size else 1.0


attributes = [str(name) for name in correlation_matrices[correlation_method].columns]
n_attributes = len(attributes)
limit = color_limit(correlation_matrices.values())

# Square plot area; the printed values shrink with the number of attributes
side_px = min(max(n_attributes * 70, 350), 700)
font_px = min(side_px / n_attributes * 0.3, 12) if n_attributes else 0

# Diverging blue-white-red palette, symmetric around 0 and shared by both methods
color_mapper = LinearColorMapper(palette=interp_palette(RdBu11, 256), low=-limit, high=limit, nan_color=NAN_COLOR)

# A categorical y axis starts at the bottom, so both axes in matrix order put the diagonal bottom-left to top-right
p = figure(x_range=FactorRange(*attributes), y_range=FactorRange(*attributes),
           frame_width=side_px, frame_height=side_px, tools="reset,save", toolbar_location="right",
           title=f"{correlation_method.capitalize()} correlation")
p.grid.grid_line_color = None
p.axis.axis_line_color = None
p.axis.major_tick_line_color = None
p.axis.major_label_text_font_size = "11pt"
p.xaxis.major_label_orientation = np.pi / 4 if n_attributes > 3 else 0

# One rect (and text) renderer per method; only the selected method is visible
layers = {}
for method, matrix in correlation_matrices.items():
    table = to_long_format(matrix, method)
    table["label"] = ["" if pd.isna(v) else f"{round(v, 2) + 0.0:.2f}" for v in table["value"]]   # + 0.0: no "-0.00"
    table["label_color"] = np.where(table["value"].abs() > DARK_CELL_SHARE * limit, "white", "black")
    source = ColumnDataSource(table)
    visible = method == correlation_method
    cells = p.rect("x", "y", width=1, height=1, source=source, visible=visible, line_color="white", line_width=2,
                   fill_color={"field": "value", "transform": color_mapper})
    layers[method] = [cells]
    if font_px >= MIN_FONT_PX:
        layers[method].append(p.text("x", "y", text="label", source=source, visible=visible,
                                     text_color=field("label_color"), text_align="center",
                                     text_baseline="middle", text_font_size=f"{font_px:.0f}px"))

p.add_tools(HoverTool(renderers=[layer[0] for layer in layers.values()],
                      tooltips=[("Attributes", "@y vs. @x"), ("Correlation", "@value{0.000}"), ("Method", "@method")]))
p.add_layout(ColorBar(color_mapper=color_mapper, title="Correlation (blue = negative, red = positive)",
                      width=18, label_standoff=8), "right")

# Dropdown that switches the visible method in the browser (no Python kernel needed)
select = Select(title="Correlation method", value=correlation_method,
                options=[(method, method.capitalize()) for method in correlation_matrices])
select.js_on_change("value", CustomJS(args=dict(plot=p, layers=layers), code="""
    const method = cb_obj.value;
    for (const [name, renderers] of Object.entries(layers)) {
        renderers.forEach(renderer => { renderer.visible = (name === method); });
    }
    const label = cb_obj.options.find(option => option[0] === method)[1];
    plot.title.text = label + " correlation";
"""))

show(column(select, p))

#### Response: Justification for colormap

Blue–white–red, going from negative to positive. White is 0, blue is negative, red is positive, and darker means stronger. Both methods use the same scale, ±0.2, because the largest off-diagonal correlation is 0.164, so colours can be compared across them. The diagonal (always 1) is fully dark red.

### Assignment V.3 (5 points)

In this assignment, we will visualize the relationship between two numerical columns and one categorical column from the dataset. We already extract the names of the first two numerical columns below; the easiest is to just use those, but instead of fixing the categorical column we now want it to be **user-selectable**, similar to Assignment V.2. (Remember that we might test this with a different dataset, so do not pick columns by name.)

The goal here is to visualize the complete data of the selected columns. We will proceed in two steps again.

In the first step, create a scatterplot to visualize the selected columns.
* A scatterplot is a common representation, bokeh provides it using `scatter`.
* Use the x and y axes for the numerical values.
* Add a `Select` dropdown listing the categorical columns in the dataset, and use it to choose which categorical column is encoded in the plot (e.g. through color). Use `js_on_change` to update the plot when a different column is selected, similar to Assignment V.2.
* Choose a proper encoding for the categorical column and **justify** it in the field below the code **in 50 words or less**. (Note: this encoding was omitted in the example figure below.)
* Make sure the chart is legible by adjusting mark sizes, adding axes, legends, a color bar, or similar, etc. It is up to you to decide what is needed.

Even with putting in a lot of effort, you will notice that there will be a lot of overplotting, even with 'just' ~50k points in the dataset. In the example below, you see iso-density lines added to show how the points distribute. We have already added the `kde` function to [estimate the density.](https://en.wikipedia.org/wiki/Multivariate_kernel_density_estimation) Use this to calculate the point density. This is taken from an [example in the Bokeh documenation](https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html). Look at the [example](https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html) and adjust it to your needs.
* Add the density information to your plot. (This only needs to depend on the two numerical columns, not on the selected categorical column.)

[<img src="assets/scatterplot.png" alt="Scatterplot with density indicator" width="700"/>](assets/scatterplot.png)


In [5]:
import warnings

from bokeh.models import CustomJSHover, Legend, LegendItem, LinearColorMapper
from bokeh.palettes import Category10_10

# kde calculation taken from https://docs.bokeh.org/en/latest/docs/examples/topics/stats/kde2d.html
# look at the example on how it is used. You can use the plot code also for inspiration, but make sure to adjust it for proper results
# note, N is the kernel size, larger values will make this lengthy to calculate but give more smooth results.
# When you use it consider using small values for N during testing.
def kde(x, y, N):

    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()

    X, Y = np.mgrid[xmin:xmax:N*1j, ymin:ymax:N*1j]
    positions = np.vstack([X.ravel(), Y.ravel()])
    values = np.vstack([x, y])
    kernel = gaussian_kde(values)
    Z = np.reshape(kernel(positions).T, X.shape)

    return X, Y, Z

# we extract the names of the first two numerical columns
num_cols = []
num_cols.append(data.select_dtypes(include="number").columns[0])
num_cols.append(data.select_dtypes(include="number").columns[1])
# and all categorical columns, to populate the dropdown.
cat_cols = list(data.select_dtypes(include=["object", "category"]).columns)
# You can just use the above columns to set up your visualization. Feel free to change these for testing.
# For the submission, you can safely assume that at least 2 numerical and 1 categorical columns are available in the dataset.


# Qualitative palette: Category10 without its grey, plus a grey for "Other" (a column with exactly ten
# categories uses it for its tenth)
PALETTE = [color for color in Category10_10 if color != "#7f7f7f"] + ["#9e9e9e"]
MISSING_LABEL = "(missing)"
MAX_LABEL_CHARS = 30           # longer category names are cut in the legend
POINT_ALPHA = 0.35
KDE_GRID = 100                 # the density is evaluated on a KDE_GRID x KDE_GRID grid
CONTOUR_FRACTIONS = np.geomspace(0.05, 0.8, 6)    # contour levels as fractions of the peak density
TOOLTIP_CATEGORIES = 4         # categories listed in a tooltip that covers several points


def encode(column):
    """Category codes of a column: (codes, categories, slot_of_code, legend_labels).

    categories are sorted by frequency (ties alphabetical) and codes index them. Every category gets a palette
    slot; if there are more categories than slots, the least frequent ones share the last slot, "Other".
    """
    text = column.astype(object).where(column.notna(), MISSING_LABEL).astype(str)
    categories = list(text.value_counts().sort_index().sort_values(ascending=False, kind="stable").index)
    codes = text.map({category: code for code, category in enumerate(categories)})
    codes = codes.to_numpy(dtype=np.min_scalar_type(len(categories)))
    n_named = len(PALETTE) if len(categories) <= len(PALETTE) else len(PALETTE) - 1
    slot_of_code = [min(code, n_named) for code in range(len(categories))]
    legend_labels = [c if len(c) <= MAX_LABEL_CHARS else c[:MAX_LABEL_CHARS - 1] + "…" for c in categories[:n_named]]
    if len(categories) > n_named:
        legend_labels.append(f"Other ({len(categories) - n_named} more)")
    return codes, categories, slot_of_code, legend_labels


def add_density_contours(fig, x, y):
    """Overlay iso-density lines of the point cloud; warn and skip if the density is degenerate."""
    try:
        X, Y, Z = kde(x, y, KDE_GRID)
    except (np.linalg.LinAlgError, ValueError) as error:  # e.g. a constant column gives a singular covariance
        warnings.warn(f"Density contours skipped: {error}")
        return
    fig.contour(X, Y, Z, Z.max() * CONTOUR_FRACTIONS, line_color="black", line_width=1, line_alpha=0.8)


num_names = [str(name) for name in num_cols]
cat_names = [str(name) for name in cat_cols]

# Rows with a missing or infinite coordinate can be neither drawn nor used for the density.
xy = data[num_cols].to_numpy(dtype=float, na_value=np.nan)
keep = np.isfinite(xy).all(axis=1)
# Fixed random draw order: otherwise the categories of the last rows would hide the others in dense regions.
order = np.random.default_rng(0).permutation(keep.sum())
x, y = xy[keep][order].T
encodings = [encode(data.loc[keep, name].iloc[order]) for name in cat_cols]
codes, categories, slot_of_code, legend_labels = map(list, zip(*encodings))

# "slot" is the palette slot of every point for the selected column; code_i are the categories of column i.
source = ColumnDataSource({"x": x, "y": y, "slot": np.array(slot_of_code[0], dtype=np.int8)[codes[0]],
                           **{f"code_{i}": column_codes for i, column_codes in enumerate(codes)}})

fig = figure(title=f"{num_names[1]} vs. {num_names[0]} (lines: density at {CONTOUR_FRACTIONS[0]:.0%}"
                   f"–{CONTOUR_FRACTIONS[-1]:.0%} of its peak)",
             frame_width=700, frame_height=500, output_backend="webgl", tools="pan,box_zoom,wheel_zoom,reset,save",
             x_axis_label=num_names[0], y_axis_label=num_names[1])
fig.xaxis.formatter = NumeralTickFormatter(format="0,0.[######]")
fig.yaxis.formatter = NumeralTickFormatter(format="0,0.[######]")

# Slot k always has colour PALETTE[k]; selecting another column only swaps the slots of the points.
mapper = LinearColorMapper(palette=PALETTE, low=-0.5, high=len(PALETTE) - 0.5)
points = fig.scatter("x", "y", source=source, size=3, line_color=None, fill_alpha=POINT_ALPHA,
                     fill_color={"field": "slot", "transform": mapper})
add_density_contours(fig, x, y)

# One legend slot per palette colour, each with an invisible one-point renderer; the callback only changes
# the labels and which slots are visible.
legend = Legend(title=cat_names[0], items=[
    LegendItem(label=legend_labels[0][k] if k < len(legend_labels[0]) else "", visible=k < len(legend_labels[0]),
               renderers=[fig.scatter([np.nan], [np.nan], fill_color=color, line_color=None)])
    for k, color in enumerate(PALETTE)])
fig.add_layout(legend, "right")

select = Select(title="Colour by", value=cat_names[0], options=cat_names, width=260)
# Tooltip of the hovered points: one point is listed in full, several are summarised by category.
# The first point speaks for all of them (Bokeh asks once per point).
tooltip = CustomJSHover(args=dict(source=source, select=select, axes=num_names, names=cat_names,
                                  categories=categories, limit=TOOLTIP_CATEGORIES), code="""
    const hits = special_vars.indices;
    if (hits.indexOf(special_vars.index) !== 0) return "";
    const fmt = (v) => v.toLocaleString("en-US", {maximumFractionDigits: 6});
    const escape = (text) => text.replace(/&/g, "&amp;").replace(/</g, "&lt;");   // tooltips are HTML
    const i = names.indexOf(select.value), code = source.data["code_" + i];
    if (hits.length === 1) {
        return escape(`${axes[0]}: ${fmt(value)}, ${axes[1]}: ${fmt(source.data.y[hits[0]])}, ${select.value}: ${categories[i][code[hits[0]]]}`);
    }
    const counts = new Map();
    for (const hit of hits) counts.set(code[hit], (counts.get(code[hit]) || 0) + 1);
    const ranked = [...counts].sort((a, b) => b[1] - a[1]);
    const listed = ranked.slice(0, limit).map(([c, n]) => `${categories[i][c]} ${n}`).join(", ");
    return escape(`${hits.length} points, ${select.value}: ${listed}` + (ranked.length > limit ? `, ${ranked.length - limit} more categories` : ""));
""")
fig.add_tools(HoverTool(renderers=[points], tooltips="@x{custom}", formatters={"@x": tooltip}))

select.js_on_change("value", CustomJS(args=dict(source=source, legend=legend, names=cat_names,
                                                slot_of_code=slot_of_code, legend_labels=legend_labels), code="""
    const i = names.indexOf(cb_obj.value);
    const code = source.data["code_" + i], slot = new Int8Array(code.length);
    for (let r = 0; r < code.length; r++) slot[r] = slot_of_code[i][code[r]];
    source.data.slot = slot;
    source.change.emit();
    legend.title = cb_obj.value;
    legend.items.forEach((item, k) => {
        const label = legend_labels[i][k];
        item.visible = label !== undefined;
        if (item.visible) item.label = {value: label};
    });
"""))

show(column(select, fig))

#### Response: Justification for categorical data encoding

Colour (hue), because the categories have no order, and size or lightness would suggest one. Up to ten categories get their own colour. With more, the nine most frequent keep theirs and the rest share grey. Points are shuffled and slightly transparent, so no category hides the others.

### Assignment V.4 (10 points)

Interactive exploration

In this final assignment we want to do some true interactive exploration, harnessing the power of Bokeh by combining selection and filtering interactions with the charts you already built.
You have two options to proceed, based on your previous results. The below descriptions are minimal solutions, feel free to go beyond the suggested functionality, if you deem it useful. If you do so add a brief list of what you added in the textfield below.

#### Pick one of the the following options:

1. Use the result of **Assignment V.1** and extend it such that clicking a bar in one of the charts affects all the charts. I.e.,
   * Make it possible to select any subset of the data by clicking the corresponding bar in any of the bar/histogram charts. E.g., clicking on the *Bachelors* bar in the *Education* chart should select all datapoints where the value of Education is Bachelors.
   * Render the effect of that selection in every chart (while keeping the full dataset visible too). E.g., render a second bar corresponding to the fraction of the corresponding points within a different color on top of the original bar.
<br/><br/>
2. Combine the results of **Assignment V.2** and **Assignment V.3** into a single linked layout.
   * Show both plots next to each other in a grid.
   * Allow selection of the numerical columns in the scatterplot by clicking on a cell in the heatmap.
   * Reuse or extend the categorical-column dropdown you already built in Assignment V.3.
   * Add a way to select one or multiple values of the categorical column and update the scatterplot based on that selection. E.g., show only the datapoints where *Education* equals *Bachelors*, when Education is selected as the categorical variable of the plot. Ensure that the axis ranges do not change when the user selects values of the categorical column so that plotted points do not shift. 

For your chosen option, explain the analysis task(s) that your interactive visualization allows for. Describe it as one or more **{action, target}** pairs. Say briefly why the interaction you built (rather than a static version of the same charts) is what makes that task possible, **in 50 words or less**.

#### Some possibly useful hints:
* Check out **TapTool** for more information on enabling clicking events on plots
* Have a look at **CDSView** for one possible way to filter out a subset of data on a plot
* [Enabling click callbacks on bars](https://docs.bokeh.org/en/2.4.3/docs/user_guide/tools.html#setting-the-active-tools)
* [Getting clicked bar index in callback functions](https://discourse.bokeh.org/t/a-different-plot-in-another-figure-when-a-bar-of-a-barchart-is-clicked/8572/2)
* [Apply hover tools to plots](https://stackoverflow.com/questions/29435200/bokeh-plotting-enable-tooltips-for-only-some-glyphs)


In [6]:
# Builds on Assignment V.1: the same small multiples (distribution_chart, bin_column), now cross-filtered.
from bokeh.layouts import row
from bokeh.models import Button, Div, TapTool
from bokeh.transform import dodge

DOMINANT_SHARE = 0.9     # a value holding more of the rows than this dominates its chart (-> log counts)
MAX_EXACT_BARS = 100     # numerical columns with more distinct values are drawn as histograms
TARGET_BINS = 40         # about this many bins per histogram
SELECTED_COLOR = "#d95f02"
CONTEXT_ALPHA = 0.5      # alpha of the bars that were not clicked in a chart that has clicked bars


def nice_bin_width(span, integers):
    """Bin width 1, 2 or 5 x 10^k that gives about TARGET_BINS bins over span."""
    raw = span / TARGET_BINS
    exponent = np.floor(np.log10(raw))
    candidates = [m * 10.0 ** e for e in (exponent, exponent + 1) for m in (1, 2, 5)]
    width = min(candidates, key=lambda c: abs(np.log(c / raw)))
    return max(width, 1.0) if integers else width


def choose_binning(series):
    """(bin_width, log_y) for a column. Numbers with many distinct values are binned (a single value would select
    next to nothing); so are numbers dominated by one value, which also get log counts as in V.1."""
    if not is_numeric(series):
        return None, False
    values = series.to_numpy(dtype=float, na_value=np.nan)
    values = values[np.isfinite(values)]
    if len(values) == 0 or values.max() == values.min():
        return None, False
    distinct, counts = np.unique(values, return_counts=True)
    dominated = counts.max() / len(values) > DOMINANT_SHARE
    if not (dominated or len(distinct) > MAX_EXACT_BARS):
        return None, False
    return nice_bin_width(values.max() - values.min(), pd.api.types.is_integer_dtype(series)), bool(dominated)


def row_codes(series, bin_width):
    """Bar index of every row in the smallest signed integer type that holds it (keeps the notebook small)."""
    bars, codes, _ = bin_column(series, bin_width)
    return codes.astype(np.min_scalar_type(-max(len(bars["count"]), 1)))


def make_selectable(chart, log_y):
    """Let clicks on a chart's bars select them and draw the selected rows of every bar on top of it.

    Linear charts get an orange bar over the full bar, which reads as the selected fraction. On a log axis the
    length of a bar is not a ratio, so only an orange line at the selected count is drawn (compare its position).
    On linear charts the line also keeps selections of under a pixel visible.
    Returns the bar data source, which holds the clicked bars in .selected.indices.
    """
    bars = chart.renderers[0]
    source, bottom, width = bars.data_source, bars.glyph.bottom, bars.glyph.width
    n_bars = len(source.data["count"])
    source.data["selected"] = np.zeros(n_bars, dtype=int)
    source.data["selected_top"] = np.full(n_bars, np.nan)      # NaN: nothing selected, nothing drawn (and no 0 on a log axis)
    bars.nonselection_glyph = bars.glyph.clone(fill_alpha=CONTEXT_ALPHA, line_alpha=CONTEXT_ALPHA)
    if not log_y:
        chart.vbar(x="x", top="selected_top", bottom=bottom, width=width, source=source,
                   fill_color=SELECTED_COLOR, line_color=None, nonselection_fill_alpha=1)
    chart.segment(x0=dodge("x", -width / 2, range=chart.x_range), x1=dodge("x", width / 2, range=chart.x_range),
                  y0="selected_top", y1="selected_top", source=source,
                  line_color=SELECTED_COLOR, line_width=3, nonselection_line_alpha=1)
    # An invisible full-height bar is what gets clicked and hovered, so that bars that are
    # less than a pixel high can still be hit.
    source.data["hit_top"] = np.full(n_bars, float(source.data["count"].max(initial=0)))
    hit = chart.vbar(x="x", top="hit_top", bottom=bottom, width=width, source=source, fill_alpha=0, line_alpha=0,
                     nonselection_fill_alpha=0, nonselection_line_alpha=0)
    chart.hover[0].renderers = [hit]
    chart.hover[0].tooltips = chart.hover[0].tooltips + [("Selected", "@selected{0,0}")]
    chart.add_tools(TapTool(renderers=[hit], mode="xor"))   # a click toggles a bar
    return source


columns = list(data.columns)
binnings = [choose_binning(data[name]) for name in columns]
charts = [distribution_chart(data[name], *binning) for name, binning in zip(columns, binnings)]
bar_sources = [make_selectable(chart, log_y) for chart, (_, log_y) in zip(charts, binnings)]
# bar index of every row in every column (the keys are the chart numbers)
row_bars = ColumnDataSource({str(i): row_codes(data[name], width)
                             for i, (name, (width, _)) in enumerate(zip(columns, binnings))})

summary = Div(text="<b>No selection</b>")
log_note = (" On the log-scale charts the selected count is the orange line: compare its position, not its length."
            if any(log_y for _, log_y in binnings) else "")
help_text = Div(text=f"""<b style="color:{BAR_COLOR}">&#9632;</b> all rows &nbsp;
    <b style="color:{SELECTED_COLOR}">&#9632;</b> selected rows. Click bars to select them, click again to deselect.
    Bars in one chart combine with OR, charts combine with AND.{log_note}""")
clear = Button(label="Clear selection", button_type="primary", width=140)

# Whenever the clicked bars of any chart change, recount the selected rows of every bar in every chart.
update = CustomJS(args=dict(sources=bar_sources, row_bars=row_bars, summary=summary), code="""
    const n_charts = sources.length;
    const bar_of = [];                       // bar_of[c][r] = bar of row r in chart c (-1: row has no number there)
    for (let c = 0; c < n_charts; c++) {
        bar_of.push(row_bars.data[c]);
    }
    const n_rows = bar_of[0].length;

    // Step 1: a row stays selected if it lies in a clicked bar of every chart that has clicked bars
    const keep_row = new Uint8Array(n_rows).fill(1);
    let active = false;                      // is anything clicked at all?
    for (let c = 0; c < n_charts; c++) {
        const clicked = sources[c].selected.indices;
        if (clicked.length === 0) {
            continue;                        // nothing clicked here: this chart does not filter
        }
        active = true;
        const bar_clicked = new Uint8Array(sources[c].data.count.length);
        for (const bar of clicked) {
            bar_clicked[bar] = 1;
        }
        for (let r = 0; r < n_rows; r++) {
            if (bar_clicked[bar_of[c][r]] !== 1) {
                keep_row[r] = 0;
            }
        }
    }

    let n_selected = 0;
    for (let r = 0; r < n_rows; r++) {
        n_selected += keep_row[r];
    }

    // Step 2: in every chart, count the selected rows per bar (the orange part of each bar)
    for (let c = 0; c < n_charts; c++) {
        const n_bars = sources[c].data.count.length;
        const selected = new Array(n_bars).fill(0);
        if (active) {
            for (let r = 0; r < n_rows; r++) {
                const bar = bar_of[c][r];
                if (keep_row[r] === 1 && bar >= 0) {
                    selected[bar] += 1;
                }
            }
        }
        const selected_top = new Array(n_bars);          // NaN = draw nothing for this bar
        for (let bar = 0; bar < n_bars; bar++) {
            selected_top[bar] = selected[bar] > 0 ? selected[bar] : NaN;
        }
        sources[c].data.selected = selected;
        sources[c].data.selected_top = selected_top;
        sources[c].change.emit();
    }

    // Step 3: the text above the charts
    if (!active) {
        summary.text = "<b>No selection</b>";
    } else if (n_selected === 0) {
        summary.text = "<b>No rows match this combination</b>";
    } else {
        const percent = (100 * n_selected / n_rows).toFixed(1);
        summary.text = "<b>" + n_selected.toLocaleString("en-US") + "</b> of " + n_rows.toLocaleString("en-US")
                       + " rows selected (" + percent + " %)";
    }
""")
for source in bar_sources:
    source.selected.js_on_change("indices", update)
# list(): the same list object in two callbacks cannot be serialized
clear.js_on_click(CustomJS(args=dict(sources=list(bar_sources)), code="for (const s of sources) s.selected.indices = [];"))

show(column(row(clear, summary), help_text, gridplot(charts, ncols=GRID_COLUMNS, toolbar_location="above")))

#### Response: Task(s) your visualization supports

{select, rows by attribute value or bin}; {compare, each attribute's distribution in the selection with all rows}; {explore, dependencies between attributes}. A static grid only shows each attribute alone. Clicking bars makes all 15 charts show the selection, e.g. the salary split of female Bachelors.

**Added beyond the minimum**
* Bars in one chart combine with OR, charts combine with AND. Clicking a selected bar again deselects it.
* A Clear button and a live count of the selected rows.
* Tooltips show selected and total counts. In a chart with clicked bars, the other bars stay visible but fainter. An orange line marks selections too small to see as a bar.
* fnlwgt, capital-gain and capital-loss are binned, so one click selects a range. On the two log-scale charts the selection is shown as an orange line, because bar length is not a ratio on a log axis.


#### AI Use

Claude (Anthropic), working as a coding agent, wrote the code for V.1–V.4 and drafted the written answers. The V.2 and V.3 cells were first drafted by sub-agents; separate reviewer agents then criticised the whole notebook, which led to fixes such as clickable tiny bars, the orange line on the log charts and the narrower heatmap colour range. The notebook was run top to bottom with the versions in environment.yml, and V.4 was clicked through in a headless browser, comparing the selected counts with pandas (Bachelors: 8,025 rows; with Female: 2,477).